# 01 — Data Acquisition

The claim this project is built to challenge is that the UK's recent run of warm summers represents a *new norm* — a stable adjustment to a slightly warmer climate. The data says something different. Not just that it's getting warmer, but that the rate of change is itself accelerating, that multiple indicators are moving simultaneously, and that the variability is increasing as well as the mean.

That is not a new norm. That is the beginning of a trajectory.

This notebook ingests 119 Met Office regional climate series files covering six variables across 17 UK regions, from as far back as 1836. It produces a single long-format processed dataset for use in all subsequent notebooks.

**Variables:**
- `Tmean`, `Tmax`, `Tmin` — mean, maximum and minimum temperature (°C)
- `Rainfall` — total precipitation (mm)
- `Sunshine` — sunshine duration (hours)
- `AirFrost` — days of air frost per period
- `Raindays1mm` — days with rainfall ≥ 1mm per period

**Source:** Met Office HadUK-Grid regional series, via https://www.metoffice.gov.uk/research/climate/maps-and-data/uk-and-regional-series

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW       = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Ingestion Function

All 119 files have identical structure: a 5-line header, then annual rows with monthly, seasonal and annual values. The variable and region are encoded in the filename — `Tmean_England_S.txt` → variable `Tmean`, region `England S`.

One function handles everything.

In [2]:
MONTH_COLS    = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
SEASONAL_COLS = ['win','spr','sum','aut']
ANNUAL_COL    = ['ann']

def parse_metoffice_file(filepath: Path) -> pd.DataFrame:
    """
    Parse a single Met Office regional series file into long format.
    
    Returns a DataFrame with columns:
        variable, region, year, period_type, period, value
    """
    stem     = filepath.stem                          # e.g. Tmean_England_S
    parts    = stem.split('_', 1)
    variable = parts[0]                               # e.g. Tmean
    region   = parts[1].replace('_', ' ')             # e.g. England S

    df = pd.read_csv(
        filepath,
        skiprows=5,
        sep=r'\s+',
        na_values=['---'],
    )

    frames = []

    # Monthly
    avail_months = [c for c in MONTH_COLS if c in df.columns]
    m = df[['year'] + avail_months].melt(id_vars='year', var_name='period', value_name='value')
    m['period_type'] = 'monthly'
    frames.append(m)

    # Seasonal
    avail_seasons = [c for c in SEASONAL_COLS if c in df.columns]
    s = df[['year'] + avail_seasons].melt(id_vars='year', var_name='period', value_name='value')
    s['period_type'] = 'seasonal'
    frames.append(s)

    # Annual
    avail_ann = [c for c in ANNUAL_COL if c in df.columns]
    a = df[['year'] + avail_ann].melt(id_vars='year', var_name='period', value_name='value')
    a['period_type'] = 'annual'
    frames.append(a)

    out = pd.concat(frames, ignore_index=True)
    out['variable'] = variable
    out['region']   = region
    out['value']    = pd.to_numeric(out['value'], errors='coerce')

    return out[['variable', 'region', 'year', 'period_type', 'period', 'value']]


print('Ingestion function ready.')

Ingestion function ready.


## Load All Files

In [3]:
txt_files = sorted(RAW.glob('*.txt'))
print(f'Files found: {len(txt_files)}')

frames = []
errors = []

for f in txt_files:
    try:
        frames.append(parse_metoffice_file(f))
    except Exception as e:
        errors.append((f.name, str(e)))
        print(f'  ERROR: {f.name} — {e}')

if errors:
    print(f'\n{len(errors)} files failed to parse.')
else:
    print('All files parsed successfully.')

Files found: 119
All files parsed successfully.


In [4]:
df = pd.concat(frames, ignore_index=True)

print(f'Total rows: {len(df):,}')
print(f'Variables: {sorted(df["variable"].unique())}')
print(f'Regions:   {sorted(df["region"].unique())}')
print(f'Year range: {df["year"].min()} – {df["year"].max()}')
print(f'Missing values: {df["value"].isna().sum():,} ({df["value"].isna().mean()*100:.1f}%)')

Total rows: 280,041
Variables: ['AirFrost', 'Raindays1mm', 'Rainfall', 'Sunshine', 'Tmax', 'Tmean', 'Tmin']
Regions:   ['East Anglia', 'England', 'England E and NE', 'England N', 'England NW and N Wales', 'England S', 'England SE and Central S', 'England SW and S Wales', 'England and Wales', 'Midlands', 'Northern Ireland', 'Scotland', 'Scotland E', 'Scotland N', 'Scotland W', 'UK', 'Wales']
Year range: 1836 – 2026
Missing values: 1,309 (0.5%)


## Coverage by Variable

Different variables have different start dates — rainfall from 1836, temperature from 1884, sunshine from 1910. This matters for trend analysis: the longer the record, the more clearly we can see whether recent change is genuinely anomalous.

In [5]:
# Annual UK series only — representative coverage check
uk_annual = df[
    (df['region'] == 'UK') & 
    (df['period'] == 'ann')
]

print('Variable coverage (UK annual series):')
print(f'{"Variable":15s} {"Start":>8} {"End":>8} {"Years":>8} {"Missing":>10}')
print('-' * 55)
for var, group in uk_annual.groupby('variable'):
    valid = group.dropna(subset=['value'])
    print(f'{var:15s} {int(valid["year"].min()):>8} {int(valid["year"].max()):>8} '
          f'{len(valid):>8} {group["value"].isna().sum():>10}')

Variable coverage (UK annual series):
Variable           Start      End    Years    Missing
-------------------------------------------------------
AirFrost            1931     2025       95          1
Raindays1mm         1891     2025      135          1
Rainfall            1836     2025      190          1
Sunshine            1910     2025      116          1
Tmax                1884     2025      142          1
Tmean               1884     2025      142          1
Tmin                1884     2025      142          1


## Sanity Checks

Verify a few known values against the raw files.

In [6]:
def check(variable, region, year, period, expected):
    val = df[
        (df['variable'] == variable) &
        (df['region']   == region)   &
        (df['year']     == year)     &
        (df['period']   == period)
    ]['value'].values
    actual = val[0] if len(val) else None
    status = 'OK' if actual == expected else f'MISMATCH (got {actual})'
    print(f'{variable} {region} {year} {period}: {actual} — {status}')

# Values read directly from raw files
check('Tmean',    'England S', 1884, 'ann', 9.72)
check('Tmean',    'England S', 2024, 'sum', None)  # expect some value
check('Rainfall', 'UK',        1836, 'ann', 1124.7)
check('Rainfall', 'UK',        1836, 'jan', 101.5)

Tmean England S 1884 ann: 9.72 — OK
Tmean England S 2024 sum: 16.17 — MISMATCH (got 16.17)
Rainfall UK 1836 ann: 1124.7 — OK
Rainfall UK 1836 jan: 101.5 — OK


## Missing Value Assessment

The `---` values in the raw files indicate periods where data is unavailable — mostly the first winter of a series (which spans two calendar years) and the most recent months of 2026 which haven't been finalised yet.

In [7]:
missing_by_period = df.groupby('period')['value'].apply(lambda x: x.isna().sum())
missing_by_period = missing_by_period[missing_by_period > 0].sort_values(ascending=False)
print('Missing values by period:')
print(missing_by_period.to_string())
print()
print('Recent years — how many months are provisional or missing?')
recent = df[(df['year'] >= 2024) & (df['period_type'] == 'monthly')]
print(recent.groupby(['year','variable'])['value'].apply(lambda x: x.isna().sum()).to_string())

Missing values by period:
period
win    238
ann    119
aug    119
aut    119
dec    119
nov    119
oct    119
sep    119
spr    119
sum    119

Recent years — how many months are provisional or missing?
year  variable   
2024  AirFrost        0
      Raindays1mm     0
      Rainfall        0
      Sunshine        0
      Tmax            0
      Tmean           0
      Tmin            0
2025  AirFrost        0
      Raindays1mm     0
      Rainfall        0
      Sunshine        0
      Tmax            0
      Tmean           0
      Tmin            0
2026  AirFrost       85
      Raindays1mm    85
      Rainfall       85
      Sunshine       85
      Tmax           85
      Tmean          85
      Tmin           85


## Save Processed Data

In [8]:
df.to_csv(PROCESSED / 'climate_series.csv', index=False)
print(f'Saved climate_series.csv — {len(df):,} rows')
print()
print('Schema:')
print(df.dtypes)
print()
print('Sample:')
print(df[
    (df['variable'] == 'Tmean') & 
    (df['region'] == 'UK') & 
    (df['period'] == 'ann')
].tail(10).to_string(index=False))

Saved climate_series.csv — 280,041 rows

Schema:
variable        object
region          object
year             int64
period_type     object
period          object
value          float64
dtype: object

Sample:
variable region  year period_type period  value
   Tmean     UK  2017      annual    ann   9.53
   Tmean     UK  2018      annual    ann   9.45
   Tmean     UK  2019      annual    ann   9.39
   Tmean     UK  2020      annual    ann   9.62
   Tmean     UK  2021      annual    ann   9.28
   Tmean     UK  2022      annual    ann  10.03
   Tmean     UK  2023      annual    ann   9.97
   Tmean     UK  2024      annual    ann   9.79
   Tmean     UK  2025      annual    ann  10.10
   Tmean     UK  2026      annual    ann    NaN


## Summary

One processed file: `climate_series.csv`

| Column | Description |
|---|---|
| `variable` | Climate variable (Tmean, Tmax, Tmin, Rainfall, Sunshine, AirFrost, Raindays1mm) |
| `region` | UK region (UK, England, Scotland, Wales, Northern Ireland, sub-regions) |
| `year` | Calendar year |
| `period_type` | monthly / seasonal / annual |
| `period` | jan–dec, win/spr/sum/aut, or ann |
| `value` | Measured value in variable-appropriate units |

**Next:** Notebook 02 — trend analysis. Is the rate of warming accelerating? The second derivative is the argument.